In [43]:
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error,root_mean_squared_error,mean_squared_error
from sklearn.model_selection import GridSearchCV
import numpy as np
from sklearn.tree import DecisionTreeRegressor

In [67]:
df = pd.read_csv("data\\train.csv")
X_all_test = pd.read_csv("data\\test.csv")
y = df['SalePrice']
Xf = df.drop(["SalePrice"],axis=1)
Xf.columns

Index(['Id', 'MSSubClass', 'MSZoning', 'LotFrontage', 'LotArea', 'Street',
       'Alley', 'LotShape', 'LandContour', 'Utilities', 'LotConfig',
       'LandSlope', 'Neighborhood', 'Condition1', 'Condition2', 'BldgType',
       'HouseStyle', 'OverallQual', 'OverallCond', 'YearBuilt', 'YearRemodAdd',
       'RoofStyle', 'RoofMatl', 'Exterior1st', 'Exterior2nd', 'MasVnrType',
       'MasVnrArea', 'ExterQual', 'ExterCond', 'Foundation', 'BsmtQual',
       'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinSF1',
       'BsmtFinType2', 'BsmtFinSF2', 'BsmtUnfSF', 'TotalBsmtSF', 'Heating',
       'HeatingQC', 'CentralAir', 'Electrical', '1stFlrSF', '2ndFlrSF',
       'LowQualFinSF', 'GrLivArea', 'BsmtFullBath', 'BsmtHalfBath', 'FullBath',
       'HalfBath', 'BedroomAbvGr', 'KitchenAbvGr', 'KitchenQual',
       'TotRmsAbvGrd', 'Functional', 'Fireplaces', 'FireplaceQu', 'GarageType',
       'GarageYrBlt', 'GarageFinish', 'GarageCars', 'GarageArea', 'GarageQual',
       'GarageCond', 'PavedDrive

In [45]:
y_scaled = (y - y.mean()) / y.std()
y_scaled

X = Xf.loc[:,Xf.isna().sum() < 1000]

In [46]:
numerical_cols = [cols for cols in X.columns
                  if X[cols].dtype in ['int64','float64']]
cardinal_cols = [cols for cols in X.columns
                 if X[cols].dtype == 'str' and X[cols].nunique() > 0]
numerical_cols = numerical_cols[1:]

In [47]:
SimpleImputerfornumericals = SimpleImputer(strategy='median')
SimpleImputerforcardiunals = SimpleImputer(strategy='most_frequent')

X[numerical_cols] = SimpleImputerfornumericals.fit_transform(X[numerical_cols])
X[cardinal_cols] = SimpleImputerforcardiunals.fit_transform(X[cardinal_cols])

In [48]:
X_scaled_numericals = X[numerical_cols].copy()

X_scaled_numericals = (X_scaled_numericals - X_scaled_numericals.mean()) / X_scaled_numericals.std()

X_scaled_numericals

,MSSubClass,LotFrontage,LotArea,OverallQual,OverallCond,YearBuilt,YearRemodAdd,MasVnrArea,BsmtFinSF1,BsmtFinSF2,...,GarageArea,WoodDeckSF,OpenPorchSF,EnclosedPorch,3SsnPorch,ScreenPorch,PoolArea,MiscVal,MoSold,YrSold
0,0.073350,-0.220799,-0.207071,0.651256,-0.517023,1.050634,0.878367,0.513928,0.575228,-0.288554,...,0.350880,-0.751918,0.216429,-0.359202,-0.116299,-0.270116,-0.068668,-0.087658,-1.598563,0.138730
1,-0.872264,0.460162,-0.091855,-0.071812,2.178881,0.156680,-0.429430,-0.570555,1.171591,-0.288554,...,-0.060710,1.625638,-0.704242,-0.359202,-0.116299,-0.270116,-0.068668,-0.087658,-0.488943,-0.614228
2,0.073350,-0.084607,0.073455,0.651256,-0.517023,0.984415,0.829930,0.325803,0.092875,-0.288554,...,0.631510,-0.751918,-0.070337,-0.359202,-0.116299,-0.270116,-0.068668,-0.087658,0.990552,0.138730
3,0.309753,-0.447787,-0.096864,0.651256,-0.517023,-1.862993,-0.720051,-0.570555,-0.499103,-0.288554,...,0.790533,-0.751918,-0.175988,4.091122,-0.116299,-0.270116,-0.068668,-0.087658,-1.598563,-1.367186
4,0.073350,0.641752,0.375020,1.374324,-0.517023,0.951306,0.733056,1.366021,0.463410,-0.288554,...,1.697903,0.779930,0.563567,-0.359202,-0.116299,-0.270116,-0.068668,-0.087658,2.100173,0.138730
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1455,0.073350,-0.356992,-0.260471,-0.071812,-0.517023,0.918196,0.733056,-0.570555,-0.972685,-0.288554,...,-0.060710,-0.751918,-0.100523,-0.359202,-0.116299,-0.270116,-0.068668,-0.087658,0.620678,-0.614228
1456,-0.872264,0.687149,0.266316,-0.071812,0.381612,0.222899,0.151813,0.087881,0.759399,0.721865,...,0.126376,2.032535,-0.704242,-0.359202,-0.116299,-0.270116,-0.068668,-0.087658,-1.598563,1.644646
1457,0.309753,-0.175402,-0.147760,0.651256,3.077516,-1.002149,1.023678,-0.570555,-0.369744,-0.288554,...,-1.033560,-0.751918,0.201336,-0.359202,-0.116299,-0.270116,-0.068668,4.951415,-0.488943,1.644646
1458,-0.872264,-0.084607,-0.080133,-0.794879,0.381612,-0.704164,0.539309,-0.570555,-0.865252,6.090101,...,-1.089686,2.168167,-0.704242,1.473284,-0.116299,-0.270116,-0.068668,-0.087658,-0.858816,1.644646


In [49]:
oh = OneHotEncoder(
    handle_unknown='ignore',
    sparse_output=False
)
X_copy = X.copy()

X_1 = pd.DataFrame(
    oh.fit_transform(X_copy[cardinal_cols]),
    columns=oh.get_feature_names_out(),
    index=X_copy.index
)

X_scaled = pd.concat([X_scaled_numericals,X_1], axis=1)

In [50]:
X_mat = X_scaled.to_numpy()
y_mat = y_scaled.to_numpy()
X_mat

array([[ 0.07334983, -0.22079943, -0.20707076, ...,  0.        ,
         1.        ,  0.        ],
       [-0.87226388,  0.46016206, -0.0918549 , ...,  0.        ,
         1.        ,  0.        ],
       [ 0.07334983, -0.08460713,  0.07345481, ...,  0.        ,
         1.        ,  0.        ],
       ...,
       [ 0.30975326, -0.175402  , -0.14775964, ...,  0.        ,
         1.        ,  0.        ],
       [-0.87226388, -0.08460713, -0.08013294, ...,  0.        ,
         1.        ,  0.        ],
       [-0.87226388,  0.2331749 , -0.05809164, ...,  0.        ,
         1.        ,  0.        ]], shape=(1460, 274))

In [51]:
class CustomGradientBoostingRegressor:
    
    def __init__(self, n_estimators, learning_rate, max_depth):
        self.n_estimators = n_estimators
        self.learning_rate = learning_rate
        self.max_depth = max_depth
        
        self.models = []
        
        self.initial_prediction = None
    
    def fit(self, X, y):
        self.initial_prediction = y.mean()
        current_predictions = np.full(shape = y.shape, fill_value = self.initial_prediction, dtype = 'float64')
        
        for i in range(self.n_estimators):
            residual = y - current_predictions
            model = DecisionTreeRegressor(max_depth = self.max_depth)
            model.fit(X, residual)
            prediction = model.predict(X)
            current_predictions += prediction * self.learning_rate
            self.models.append(model)
            
    def predict(self, X):
        final_predictions = np.full(shape=X.shape[0], fill_value=self.initial_prediction, dtype='float64')
        
        for model in self.models:
            predict = model.predict(X)
            final_predictions += predict * self.learning_rate
        return final_predictions

In [63]:
my_gbm = CustomGradientBoostingRegressor(n_estimators=1000, learning_rate=0.1, max_depth=3)

my_gbm.fit(X_mat,y_mat)

gbm_predictions = my_gbm.predict(X_mat)

rmse = np.sqrt(np.mean( ((gbm_predictions - y_mat)**2)))

actual_error_in_doll = rmse * y.std()

In [64]:
actual_error_in_doll

np.float64(3536.0246395748836)

In [53]:
models = []
learning_rate = 0.01

def fit(X, y, learning_rate, n_estimators, max_depth):
    initial_pred = y.mean()
    current_pred = np.full(shape = y.shape, fill_value = initial_pred, dtype = 'float64')
    
    for i in range(n_estimators):
        residual = y - current_pred
        model = DecisionTreeRegressor(max_depth = max_depth)
        model.fit(X, residual)
        pred = model.predict(X)
        current_pred += pred * learning_rate
        models.append(model)
    return initial_pred
    
def predict(X, initial_pred, learning_rate):
        final_pred = np.full(shape = X.shape[0], fill_value = initial_pred, dtype = 'float64')
        
        for model in models:
            pred = model.predict(X)
            final_pred += pred * learning_rate
        
        return final_pred
    
initial_pred = fit(X = X_mat, y = y_mat, learning_rate = learning_rate, n_estimators = 10000, max_depth = 3)
pred = predict(X = X_mat, initial_pred = initial_pred, learning_rate = learning_rate)

In [65]:
rmse = np.sqrt(np.mean((pred - y_mat)**2))
actual_error_in_doll = rmse * y.std()
actual_error_in_doll

np.float64(3785.4252274340715)